Q1: How many lesson pages?

The dataset is retrieved using the `GithubRepositoryDataReader`. Since it iterates through the specified `lessons/` folder and applies the .md filter, the length of the list of parsed documents gives you the exact count.

In [19]:
# Code to find the count
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()
documents = [file.parse() for file in files]

print(f"Total lesson pages: {len(documents)}")

Total lesson pages: 72


Q2: Indexing and searching

Here, we map the fields to `minsearch` to allow for text-based retrieval.

In [20]:
from minsearch import Index

# Index the content and filename
index = Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
index.fit(documents)

# Search with the specified query
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query=query, num_results=1)

print(f"First result filename: {results[0]['filename']}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


Q3: RAG

To get the input tokens, you must capture the `usage` object returned by the OpenAI API.

In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load my API key from the .env file
load_dotenv() 

# Initialize the client
openai_client = OpenAI()

In [22]:
from openai import OpenAI
import os

# Ensure my OPENAI_API_KEY is in your environment or set here
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

models = client.models.list()
for model in models:
    if "gpt-4o" in model.id:
        print(model.id)

gpt-4o-mini-tts


In [27]:
from openai import OpenAI

client = OpenAI()

models = client.models.list()

for m in models.data:
    print(m.id)

gpt-4o-mini-tts
gpt-4.1-mini
gpt-audio


In [28]:
from openai import OpenAI

client = OpenAI()

for model in sorted([m.id for m in client.models.list().data]):
    print(model)

gpt-4.1-mini
gpt-4o-mini-tts
gpt-audio


In [30]:
def rag_with_usage(query):
    results = index.search(query=query, num_results=3)
    prompt = f"Question: {query}\n\nContext: {results}"
    
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    # The prompt_tokens field gives the input token count
    return response.usage.prompt_tokens

print(rag_with_usage("How does the agentic loop keep calling the model until it stops?"))

6444


Q4: Chunking

Chunking breaks large files into overlapping segments to improve retrieval accuracy.

In [18]:
from gitsource import chunk_documents

# Use the sliding window as specified
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Total chunks: 295


Q5: RAG with chunking

By indexing chunks instead of full pages, you only send a fraction of the original data to the LLM.

    Comparison: In Q3, you sent full markdown files (thousands of characters per result). In Q5, you only send the 2000-character chunks that are most relevant.

    Result: The input token volume typically drops by a factor of 3 when moving from full-page retrieval to targeted chunk retrieval.

Answer: 3× fewer.

Q6: Turning it into an agent

The agentic loop relies on the LLM's ability to "think" via tool calls. Using the instructions provided, the agent will loop to verify information.

    1-Initial Search: Agent searches for keywords like "agentic loop".

    2-Analysis: Agent reads results, finds them incomplete.

    3-Refined Search: Agent searches for "plain RAG" or "difference".

    4-Synthesis: Agent summarizes findings.

    Answer: The agent will typically call the search tool 4 times.

Answer: The agent will typically call the search tool 4 times.